In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ BCE = L(y) = -(y\ln(p)+(1-y)\ln(1-p)) $$
$$ \frac{\partial L}{\partial p} = \frac{p-y}{p(1-p)} $$
$$ \frac{\partial L}{\partial w} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} \frac{\partial z}{\partial w} = \frac{p-y}{p(1-p)} \, p(1-p) \, x = (p-y)x $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial p} \frac{\partial p}{\partial z} \frac{\partial z}{\partial b} = \frac{p-y}{p(1-p)} \, p(1-p) \, 1 = (p-y)1 $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from common import assert_eq, T # type: ignore


def binary_cross_entropy(probs: tr.Tensor, y: tr.Tensor, reduction:str = "mean") -> tr.Tensor:
    probs = probs.clamp(0.0001, 0.9999)
    bce = - (y * probs.log() + (1 - y) * (1 - probs).log())
    
    if reduction == "mean":
        return bce.mean()
    elif reduction == "sum":
        return bce.sum()
    else:
        return bce


class BinaryCrossEntropyFunction(autograd.Function):
    @staticmethod
    def forward(ctx, probs: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
        probs = probs.clamp(0.0001, 0.9999)
        ctx.save_for_backward(probs, y)
        return binary_cross_entropy(probs, y)

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, None]:
        (probs, y) = ctx.saved_tensors

        N = probs.size(0)
        df_dp = (probs - y) / (probs * (1 - probs))
        dF_dp = dF_df * df_dp / N

        # y is treated as constant label → no gradient
        dF_dy = None

        return (dF_dp, dF_dy)
   

# `BinaryCrossEntropyFunction` implements the actual operator.
# `BinaryCrossEntropy(Module)` is just a wrapper for nicer API.
class BinaryCrossEntropy(nn.Module):
    def forward(self, probs: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
        return BinaryCrossEntropyFunction.apply(probs, y)


def test_binary_cross_entropy():
    tr.manual_seed(0)

    samples = 10
    p = tr.rand(samples, 1, requires_grad=True)
    y = tr.randint(0, 2, (samples, 1)).float()

    p1 = p.clone().detach().requires_grad_(True)
    y1 = BinaryCrossEntropy()(p1, y)
    y1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    y2 = nn.BCELoss()(p2, y)
    y2.backward()

    assert_eq(y1.item(), y2.item(), atol=1e-4)
    assert_eq(p1.grad, p2.grad, atol=1e-4)


if __name__ == "__main__":
    test_binary_cross_entropy()